# Update market data in Cloudflare D1

Use this after the local IDX database has new imported market data. This notebook uploads only the latest 21 price dates, the newest small reference datasets, and cached fundamentals. It never uploads users, preferences, watchlists, Telegram queue/history, tokens, or backtest results.

The proxy token is requested privately with `getpass`; it is not printed or saved in the notebook.

## 1) Create the sanitized update file on your PC

From the repository folder, run in PowerShell:
```powershell
& "$env:LOCALAPPDATA\Programs\Python\Python312\python.exe" pkg/tools/export_d1_update.py
```
Then select the generated `.json.gz` file from `.d1_exports` below. Do not upload the SQLite database itself.

In [ ]:
from google.colab import files
uploaded = files.upload()
assert len(uploaded) == 1, 'Pilih tepat satu file .json.gz'
filename, compressed = next(iter(uploaded.items()))
assert filename.endswith('.json.gz'), 'File harus berakhiran .json.gz'
print('File diterima:', filename, '| ukuran:', len(compressed), 'bytes')

In [ ]:
import gzip, json
bundle = json.loads(gzip.decompress(compressed).decode('utf-8'))
assert bundle.get('format_version') == 2, 'Format bundle tidak dikenal'
allowed = {'shared_datasets', 'shared_records', 'fundamentals_cache', 'app_meta'}
assert set(bundle['tables']).issubset(allowed), 'Bundle memuat tabel yang tidak diizinkan'
assert bundle['tables']['shared_datasets']['rows'], 'Bundle tidak berisi dataset'
print('Tanggal harga:', ', '.join(bundle['px_dates']))
print('Jumlah baris yang akan diproses:')
for name, table in bundle['tables'].items(): print(f"  {name}: {len(table['rows'])}")
print('Total bundle:', bundle['source_rows'])

## 2) Check access and the plan

This checks for conflicting versions before writing. If it reports a hash/count mismatch, stop and review the source data; the notebook will not overwrite that snapshot.

In [ ]:
import getpass, requests
BASE_URL = 'https://idx-screener-state.sumarni.workers.dev'
TOKEN = getpass.getpass('D1 proxy token (input hidden): ')
HEADERS = {'Authorization': 'Bearer ' + TOKEN, 'Content-Type': 'application/json'}
def query(sql, params=()):
    r = requests.post(BASE_URL + '/api/query', headers=HEADERS, json={'query': sql, 'params': list(params)}, timeout=60)
    if r.status_code != 200: raise RuntimeError(f'Worker query gagal (HTTP {r.status_code}); periksa token/jaringan.')
    data = r.json()
    if not data.get('success'): raise RuntimeError('D1 query gagal: ' + str(data.get('error', 'unknown error')))
    return data.get('results', [])
def execute_batch(statements):
    if not statements: return
    r = requests.post(BASE_URL + '/api/batch', headers=HEADERS, json={'statements': statements}, timeout=90)
    if r.status_code != 200: raise RuntimeError(f'Worker write gagal (HTTP {r.status_code}); periksa kuota D1/jaringan.')
    data = r.json()
    if not data.get('success'): raise RuntimeError('D1 write gagal: ' + str(data.get('error', 'unknown error')))
    return data
print('D1 tersambung. Dataset saat ini:', query('SELECT COUNT(*) AS n FROM shared_datasets')[0]['n'])

In [ ]:
from collections import defaultdict
tables = bundle['tables']
ds_cols = tables['shared_datasets']['columns']; ds_rows = tables['shared_datasets']['rows']
ds_pos = {name: ds_cols.index(name) for name in ds_cols}
selected_types = sorted({row[ds_pos['dataset_type']] for row in ds_rows})
assert set(selected_types).issubset({'px', 'days', 'tipe', 'sat', 'kl', 'nm', 'klc'}), 'Ada tipe dataset yang tidak diizinkan'
marks = ','.join('?' for _ in selected_types)
existing = query(f'SELECT id, dataset_type, data_date, content_sha256 FROM shared_datasets WHERE dataset_type IN ({marks})', selected_types)
by_key = {(r['dataset_type'], r['data_date']): r for r in existing}
records_by_source = defaultdict(list)
rec_cols = tables['shared_records']['columns']; rec_pos = {name: rec_cols.index(name) for name in rec_cols}
for row in tables['shared_records']['rows']: records_by_source[row[rec_pos['dataset_id']]].append(row)
replace_source_ids = set()
for row in ds_rows:
    key = (row[ds_pos['dataset_type']], row[ds_pos['data_date']])
    current = by_key.get(key)
    expected = len(records_by_source.get(row[ds_pos['id']], []))
    if current:
        actual = query('SELECT COUNT(*) AS n FROM shared_records WHERE dataset_id=?', [current['id']])[0]['n']
        if current['content_sha256'] != row[ds_pos['content_sha256']] or actual != expected:
            replace_source_ids.add(row[ds_pos['id']])
print('Preflight lolos. Dataset baru:', sum((r[ds_pos['dataset_type']], r[ds_pos['data_date']]) not in by_key for r in ds_rows))
print('Snapshot berbeda/parsial yang akan disinkronkan ulang:', len(replace_source_ids))
print('Snapshot yang identik akan dilewati:', len(ds_rows) - sum((r[ds_pos['dataset_type']], r[ds_pos['data_date']]) not in by_key for r in ds_rows) - len(replace_source_ids))

## 3) Apply and verify

This is idempotent: rerunning the same bundle will not duplicate snapshots. It keeps only the newest 21 price dates in D1 to control database size.

In [ ]:
new_rows = [r for r in ds_rows if (r[ds_pos['dataset_type']], r[ds_pos['data_date']]) not in by_key]
dataset_insert = 'INSERT OR IGNORE INTO shared_datasets (dataset_type, source_filename, data_date, content_sha256, row_count, imported_at, imported_by, origin, parser_version) VALUES (?,?,?,?,?,?,NULL,?,?)'
for start in range(0, len(new_rows), 20):
    statements = []
    for row in new_rows[start:start+20]:
        p = ds_pos
        statements.append({'query': dataset_insert, 'params': [row[p['dataset_type']], 'market-data-update', row[p['data_date']], row[p['content_sha256']], row[p['row_count']], row[p['imported_at']], 'colab_update', row[p['parser_version']]]})
    execute_batch(statements)
cloud = query(f'SELECT id, dataset_type, data_date, content_sha256 FROM shared_datasets WHERE dataset_type IN ({marks})', selected_types)
cloud_by_key = {(r['dataset_type'], r['data_date']): r for r in cloud}
source_to_cloud = {}
for row in ds_rows:
    key = (row[ds_pos['dataset_type']], row[ds_pos['data_date']])
    source_to_cloud[row[ds_pos['id']]] = cloud_by_key[key]['id']
new_source_ids = {row[ds_pos['id']] for row in new_rows}
sync_source_ids = new_source_ids | replace_source_ids
for source_id in replace_source_ids:
    execute_batch([{'query': 'DELETE FROM shared_records WHERE dataset_id=?', 'params': [source_to_cloud[source_id]]}])
pending_records = []
for source_id in sync_source_ids:
    for row in records_by_source[source_id]:
        pending_records.append((source_to_cloud[source_id], row[rec_pos['security_code']], row[rec_pos['record_date']], row[rec_pos['payload_json']]))
record_sql = 'INSERT OR IGNORE INTO shared_records (dataset_id, security_code, record_date, payload_json) VALUES ' + ','.join('(?,?,?,?)' for _ in range(20))
for start in range(0, len(pending_records), 400):
    statements = []
    for chunk_start in range(start, min(start+400, len(pending_records)), 20):
        chunk = pending_records[chunk_start:chunk_start+20]
        sql = record_sql if len(chunk) == 20 else 'INSERT OR IGNORE INTO shared_records (dataset_id, security_code, record_date, payload_json) VALUES ' + ','.join('(?,?,?,?)' for _ in chunk)
        statements.append({'query': sql, 'params': [v for record in chunk for v in record]})
    execute_batch(statements)
fund_cols = tables['fundamentals_cache']['columns']; fund_pos = {name: fund_cols.index(name) for name in fund_cols}
fund_rows = tables['fundamentals_cache']['rows']
fund_sql = 'INSERT INTO fundamentals_cache (security_code, fetched_at, payload_json) VALUES (?,?,?) ON CONFLICT(security_code) DO UPDATE SET fetched_at=excluded.fetched_at, payload_json=excluded.payload_json'
for start in range(0, len(fund_rows), 50):
    execute_batch([{'query': fund_sql, 'params': [r[fund_pos['security_code']], r[fund_pos['fetched_at']], r[fund_pos['payload_json']]]} for r in fund_rows[start:start+50]])
meta = tables.get('app_meta', {'columns': [], 'rows': []})
if meta['rows']:
    mp = {name: meta['columns'].index(name) for name in meta['columns']}
    meta_sql = 'INSERT INTO app_meta (key, value) VALUES (?,?) ON CONFLICT(key) DO UPDATE SET value=excluded.value'
    execute_batch([{'query': meta_sql, 'params': [r[mp['key']], r[mp['value']]]} for r in meta['rows']])
for row in ds_rows:
    key = (row[ds_pos['dataset_type']], row[ds_pos['data_date']]); cloud_id = source_to_cloud[row[ds_pos['id']]]
    actual = query('SELECT COUNT(*) AS n FROM shared_records WHERE dataset_id=?', [cloud_id])[0]['n']
    expected = len(records_by_source.get(row[ds_pos['id']], []))
    if actual != expected: raise RuntimeError(f"Verifikasi gagal {key}: D1={actual}, sumber={expected}")
metadata_sql = 'UPDATE shared_datasets SET source_filename=?, content_sha256=?, row_count=?, imported_at=?, imported_by=NULL, origin=?, parser_version=? WHERE id=?'
changed = new_source_ids | replace_source_ids
for start in range(0, len(changed), 20):
    changed_rows = [row for row in ds_rows if row[ds_pos['id']] in changed][start:start+20]
    execute_batch([{'query': metadata_sql, 'params': ['market-data-update', r[ds_pos['content_sha256']], r[ds_pos['row_count']], r[ds_pos['imported_at']], 'colab_update', r[ds_pos['parser_version']], source_to_cloud[r[ds_pos['id']]]]} for r in changed_rows])
execute_batch([{'query': "DELETE FROM shared_datasets WHERE dataset_type='px' AND id NOT IN (SELECT id FROM shared_datasets WHERE dataset_type='px' ORDER BY data_date DESC LIMIT 21)"}])
print('Selesai. Snapshot px:', query("SELECT COUNT(*) AS n FROM shared_datasets WHERE dataset_type='px'")[0]['n'])
print('Total dataset:', query('SELECT COUNT(*) AS n FROM shared_datasets')[0]['n'])
print('Total record market:', query('SELECT COUNT(*) AS n FROM shared_records')[0]['n'])
print('Cache fundamental:', query('SELECT COUNT(*) AS n FROM fundamentals_cache')[0]['n'])
print('Akun, watchlist, antrean/notifikasi, token, dan backtest tidak disentuh.')
TOKEN = None
HEADERS = None